In [18]:
# -*- coding: utf-8 -*-
"""
Version SANS AUCUNE DÉPENDANCE EXTERNE (ni pandas, ni numpy) — tout est fait
avec les modules standards de Python, déjà inclus avec IDLE. Rien à installer.

Transforme un fichier journalier Météo-France brut (AAAAMMJJ;...;TN;...;TX;...)
en data/entzheim.json pour le site.

MARCHE À SUIVRE DANS IDLE :
1. File > Open > sélectionne ce fichier.
2. Vérifie la ligne CHEMIN_FICHIER ci-dessous (même dossier que ton CSV, ou
   chemin complet).
3. Run > Run Module (ou F5).
4. Regarde la fenêtre Python Shell qui s'ouvre : elle affiche les colonnes
   détectées, puis le résultat final.
"""

import csv
import json
from datetime import date
from collections import defaultdict

# ----------------------------------------------------------------------
# 1) CONFIGURATION — adapte si besoin
# ----------------------------------------------------------------------
CHEMIN_FICHIER = r"C:\Users\PaulineJoly\agence du climat, le guichet des solutions\MOBILITE DECARBONEE - Documents\DONNEES\AUTRES_ANALYSES\CLIMAT\METEO_FRANCE\donnees_67\Q_67_previous-1950-2024_RR-T-2026Vent.csv"
\Q_67_previous-1950-2024_RR-T-Vent.csv"
SEPARATEUR = ";"          # essaie "," si le fichier ne se découpe pas bien
PERIODE_REF = (1976, 2005)

# ----------------------------------------------------------------------
# 2) LECTURE DU FICHIER
# ----------------------------------------------------------------------
def vers_nombre(valeur):
    """Convertit une valeur texte en nombre, ou None si vide/invalide."""
    if valeur is None or valeur.strip() == "":
        return None
    try:
        return float(valeur.replace(",", "."))  # au cas où décimales avec virgule
    except ValueError:
        return None

print(f"Lecture de {CHEMIN_FICHIER} ...")
with open(CHEMIN_FICHIER, encoding="utf-8", errors="replace") as f:
    lecteur = csv.DictReader(f, delimiter=SEPARATEUR)
    colonnes = lecteur.fieldnames
    print("Colonnes détectées :", colonnes)
    # 👉 Vérifie ici que AAAAMMJJ, TX, TN apparaissent bien séparément.
    #    Si tu vois une seule grosse colonne, change SEPARATEUR ci-dessus.

    lignes_brutes = list(lecteur)

print(f"{len(lignes_brutes)} lignes lues.")
print("Aperçu de la première ligne :", lignes_brutes[0] if lignes_brutes else "AUCUNE")

# ----------------------------------------------------------------------
# 3) TRANSFORMATION : une entrée par jour, avec les champs utiles
# ----------------------------------------------------------------------
jours = []
for ligne in lignes_brutes:
    brut = ligne.get("AAAAMMJJ", "").strip()
    if len(brut) != 8:
        continue
    try:
        d = date(int(brut[0:4]), int(brut[4:6]), int(brut[6:8]))
    except ValueError:
        continue

    tx = vers_nombre(ligne.get("TX"))
    tn = vers_nombre(ligne.get("TN"))
    tm = vers_nombre(ligne.get("TM"))
    if tm is None and tx is not None and tn is not None:
        tm = (tx + tn) / 2

    jours.append({
        "date": d,
        "annee": d.year,
        "mois": d.month,
        "jour_annee": d.timetuple().tm_yday,
        "tx": tx,
        "tn": tn,
        "tm": tm,
    })

print(f"{len(jours)} jours exploitables après nettoyage.")

# ----------------------------------------------------------------------
# 4) SEUIL DE VAGUE DE CHALEUR : centile 95 de TM par jour de l'année,
#    calculé sur la période de référence (méthode DRIAS simplifiée)
# ----------------------------------------------------------------------
def centile_95(valeurs):
    valeurs = sorted(v for v in valeurs if v is not None)
    if not valeurs:
        return None
    index = round(0.95 * (len(valeurs) - 1))
    return valeurs[index]

valeurs_par_jour_annee = defaultdict(list)
for j in jours:
    if PERIODE_REF[0] <= j["annee"] <= PERIODE_REF[1]:
        valeurs_par_jour_annee[j["jour_annee"]].append(j["tm"])

seuil_par_jour_annee = {
    jour_annee: centile_95(valeurs)
    for jour_annee, valeurs in valeurs_par_jour_annee.items()
}

for j in jours:
    seuil = seuil_par_jour_annee.get(j["jour_annee"])
    j["jour_chaud_ref"] = (seuil is not None and j["tm"] is not None and j["tm"] > seuil)

# ----------------------------------------------------------------------
# 5) AGRÉGATION ANNUELLE
# ----------------------------------------------------------------------
def moyenne(valeurs):
    valeurs = [v for v in valeurs if v is not None]
    return round(sum(valeurs) / len(valeurs), 2) if valeurs else None

def compter_jours_vague(jours_annee):
    jours_annee = sorted(jours_annee, key=lambda j: j["date"])
    total, run = 0, 0
    for j in jours_annee:
        if j["jour_chaud_ref"]:
            run += 1
        else:
            if run >= 3:
                total += run
            run = 0
    if run >= 3:
        total += run
    return total

jours_par_annee = defaultdict(list)
for j in jours:
    jours_par_annee[j["annee"]].append(j)

annees_stats = []
for annee in sorted(jours_par_annee):
    jours_annee = jours_par_annee[annee]
    ete = [j for j in jours_annee if j["mois"] in (6, 7, 8)]

    annees_stats.append({
        "annee": annee,
        "tx_moy": moyenne([j["tx"] for j in jours_annee]),
        "tn_moy": moyenne([j["tn"] for j in jours_annee]),
        "ete_tm_moy": moyenne([j["tm"] for j in ete]),
        "jours_chauds_25": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 25),
        "jours_chauds_30": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 30),
        "nuits_tropicales": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] >= 20),
        "jours_gel": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] < 0),
        "jours_vague_chaleur": compter_jours_vague(jours_annee),
    })

# ----------------------------------------------------------------------
# 6) EXPORT JSON
# ----------------------------------------------------------------------
payload = {
    "station": "Strasbourg-Entzheim",
    "periode_reference": f"{PERIODE_REF[0]}-{PERIODE_REF[1]}",
    "source": "Météo-France",
    "projections": None,
    "annees": annees_stats,
}

with open("entzheim.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"\n✅ entzheim.json généré : {len(annees_stats)} années "
      f"({annees_stats[0]['annee']}–{annees_stats[-1]['annee']})")
print("   Déplace ce fichier dans data/entzheim.json à la racine du projet.")


SyntaxError: unexpected character after line continuation character (2695633801.py, line 27)

In [22]:
#POUR LE BAS RHIN
# -*- coding: utf-8 -*-
"""
Un fichier Météo-France "départemental" (ex. Q_67_previous-1950-2024_RR-T-Vent.csv)
contient TOUTES les stations du département, pas une seule. Ce script les
détecte automatiquement et génère un data/<station>.json pour chacune.

Sans dépendance externe (aucun pip/conda requis) — modules standards seulement.

MARCHE À SUIVRE :
1. Vérifie CHEMIN_FICHIER ci-dessous.
2. Lance le script (IDLE : F5, ou Jupyter : Shift+Entrée).
3. Il affiche d'abord la liste des stations trouvées avec leur période de
   données disponible — regarde si celles qui t'intéressent y sont.
4. Il génère ensuite un fichier JSON par station dans le dossier courant.
"""

import csv
import json
import re
from datetime import date
from collections import defaultdict

CHEMIN_FICHIER = r"C:\Users\PaulineJoly\agence du climat, le guichet des solutions\MOBILITE DECARBONEE - Documents\DONNEES\AUTRES_ANALYSES\CLIMAT\METEO_FRANCE\donnees_67\Q_67_previous-1950-2024_RR-T-2026Vent.csv"
SEPARATEUR = ";"
PERIODE_REF = (1976, 2005)
MIN_ANNEES = 50   # ignore les stations avec trop peu d'historique pour être utiles

# ----------------------------------------------------------------------
def vers_nombre(valeur):
    if valeur is None or str(valeur).strip() == "":
        return None
    try:
        return float(str(valeur).replace(",", "."))
    except ValueError:
        return None

def slugifier(nom):
    """'STRASBOURG-ENTZHEIM' -> 'strasbourg-entzheim' (utilisable comme nom de fichier/id)"""
    nom = nom.strip().lower()
    nom = re.sub(r"[^a-z0-9]+", "-", nom)
    return nom.strip("-")

def centile_95(valeurs):
    valeurs = sorted(v for v in valeurs if v is not None)
    if not valeurs:
        return None
    index = round(0.95 * (len(valeurs) - 1))
    return valeurs[index]

def moyenne(valeurs):
    valeurs = [v for v in valeurs if v is not None]
    return round(sum(valeurs) / len(valeurs), 2) if valeurs else None

def compter_jours_vague(jours_annee):
    jours_annee = sorted(jours_annee, key=lambda j: j["date"])
    total, run = 0, 0
    for j in jours_annee:
        if j["jour_chaud_ref"]:
            run += 1
        else:
            if run >= 3:
                total += run
            run = 0
    if run >= 3:
        total += run
    return total

# ----------------------------------------------------------------------
# 1) LECTURE + regroupement par station
# ----------------------------------------------------------------------
print(f"Lecture de {CHEMIN_FICHIER} ...")
with open(CHEMIN_FICHIER, encoding="utf-8", errors="replace") as f:
    lecteur = csv.DictReader(f, delimiter=SEPARATEUR)
    colonnes = lecteur.fieldnames
    print("Colonnes détectées :", colonnes)
    lignes_brutes = list(lecteur)

print(f"{len(lignes_brutes)} lignes lues.\n")

# Colonnes habituelles Météo-France : NUM_POSTE, NOM_USUEL identifient la station
jours_par_station = defaultdict(list)

for ligne in lignes_brutes:
    brut = ligne.get("AAAAMMJJ", "").strip()
    if len(brut) != 8:
        continue
    try:
        d = date(int(brut[0:4]), int(brut[4:6]), int(brut[6:8]))
    except ValueError:
        continue

    nom_station = (ligne.get("NOM_USUEL") or ligne.get("NUM_POSTE") or "STATION_INCONNUE").strip()

    tx = vers_nombre(ligne.get("TX"))
    tn = vers_nombre(ligne.get("TN"))
    tm = vers_nombre(ligne.get("TM"))
    if tm is None and tx is not None and tn is not None:
        tm = (tx + tn) / 2

    jours_par_station[nom_station].append({
        "date": d, "annee": d.year, "mois": d.month,
        "jour_annee": d.timetuple().tm_yday,
        "tx": tx, "tn": tn, "tm": tm,
    })

# ----------------------------------------------------------------------
# 2) APERÇU DES STATIONS TROUVÉES
# ----------------------------------------------------------------------
print("="*70)
print("STATIONS TROUVÉES DANS CE FICHIER")
print("="*70)
stations_retenues = []
for nom_station, jours in sorted(jours_par_station.items()):
    annees = sorted(set(j["annee"] for j in jours))
    nb_annees = len(annees)
    marque = "" if nb_annees >= MIN_ANNEES else "  (ignorée, historique trop court)"
    print(f"- {nom_station:35s} {annees[0]}-{annees[-1]}  ({nb_annees} années){marque}")
    if nb_annees >= MIN_ANNEES:
        stations_retenues.append(nom_station)

print(f"\n{len(stations_retenues)} station(s) seront traitées.\n")

# ----------------------------------------------------------------------
# 3) TRAITEMENT DE CHAQUE STATION RETENUE
# ----------------------------------------------------------------------
recap = []
for nom_station in stations_retenues:
    jours = jours_par_station[nom_station]

    # seuil vague de chaleur (centile 95 sur période de référence, par station)
    valeurs_par_jour_annee = defaultdict(list)
    for j in jours:
        if PERIODE_REF[0] <= j["annee"] <= PERIODE_REF[1]:
            valeurs_par_jour_annee[j["jour_annee"]].append(j["tm"])
    seuil_par_jour_annee = {ja: centile_95(v) for ja, v in valeurs_par_jour_annee.items()}
    for j in jours:
        seuil = seuil_par_jour_annee.get(j["jour_annee"])
        j["jour_chaud_ref"] = (seuil is not None and j["tm"] is not None and j["tm"] > seuil)

    jours_par_annee = defaultdict(list)
    for j in jours:
        jours_par_annee[j["annee"]].append(j)

    annees_stats = []
    for annee in sorted(jours_par_annee):
        jours_annee = jours_par_annee[annee]
        ete = [j for j in jours_annee if j["mois"] in (6, 7, 8)]
        annees_stats.append({
            "annee": annee,
            "tx_moy": moyenne([j["tx"] for j in jours_annee]),
            "tn_moy": moyenne([j["tn"] for j in jours_annee]),
            "ete_tm_moy": moyenne([j["tm"] for j in ete]),
            "jours_chauds_25": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 25),
            "jours_chauds_30": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 30),
            "nuits_tropicales": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] >= 20),
            "jours_gel": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] < 0),
            "jours_vague_chaleur": compter_jours_vague(jours_annee),
        })

    slug = slugifier(nom_station)
    payload = {
        "station": nom_station.title(),
        "periode_reference": f"{PERIODE_REF[0]}-{PERIODE_REF[1]}",
        "source": "Météo-France",
        "projections": None,
        "annees": annees_stats,
    }
    nom_fichier = f"{slug}.json"
    with open(nom_fichier, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    recap.append((nom_station, slug, nom_fichier, len(annees_stats)))
    print(f"✅ {nom_fichier} généré ({len(annees_stats)} années) — id à utiliser dans communes.json : \"{slug}\"")

print("\n" + "="*70)
print("RÉCAPITULATIF — à reporter dans data/communes.json (liste 'stations')")
print("="*70)
for nom_station, slug, nom_fichier, n in recap:
    print(f'{{ "id": "{slug}", "nom": "{nom_station.title()}", "fichier": "data/{nom_fichier}" }}')

Lecture de C:\Users\PaulineJoly\agence du climat, le guichet des solutions\MOBILITE DECARBONEE - Documents\DONNEES\AUTRES_ANALYSES\CLIMAT\METEO_FRANCE\donnees_67\Q_67_previous-1950-2024_RR-T-2026Vent.csv ...
Colonnes détectées : ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM', 'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50', 'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY', 'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI', 'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S', 'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR', 'STATUS_FXI3S', 'STATUS_DXI3S']
794203 lignes lues.

STATIONS TROUVÉES DANS CE FICHIER
- ALTECKENDORF                        1955-2010  (56 années)
- BARR                                1950-2018  (69 années)
- BELMONT                             1977-2024  (45 années)  (ignorée, historique tr

✅ alteckendorf.json généré (56 années) — id à utiliser dans communes.json : "alteckendorf"
✅ barr.json généré (69 années) — id à utiliser dans communes.json : "barr"
✅ brumath.json généré (75 années) — id à utiliser dans communes.json : "brumath"
✅ ebersheim.json généré (75 années) — id à utiliser dans communes.json : "ebersheim"
✅ haguenau.json généré (75 années) — id à utiliser dans communes.json : "haguenau"
✅ le-hohwald.json généré (68 années) — id à utiliser dans communes.json : "le-hohwald"
✅ preuschdorf.json généré (62 années) — id à utiliser dans communes.json : "preuschdorf"
✅ saales.json généré (52 années) — id à utiliser dans communes.json : "saales"
✅ saverne.json généré (63 années) — id à utiliser dans communes.json : "saverne"
✅ stattmatten.json généré (63 années) — id à utiliser dans communes.json : "stattmatten"
✅ strasbourg-botanique.json généré (75 années) — id à utiliser dans communes.json : "strasbourg-botanique"
✅ strasbourg-entzheim.json généré (75 années) — id à 

In [23]:
#POUR LE HAUT RHIN
# -*- coding: utf-8 -*-
"""
Un fichier Météo-France "départemental" (ex. Q_67_previous-1950-2024_RR-T-Vent.csv)
contient TOUTES les stations du département, pas une seule. Ce script les
détecte automatiquement et génère un data/<station>.json pour chacune.

Sans dépendance externe (aucun pip/conda requis) — modules standards seulement.

MARCHE À SUIVRE :
1. Vérifie CHEMIN_FICHIER ci-dessous.
2. Lance le script (IDLE : F5, ou Jupyter : Shift+Entrée).
3. Il affiche d'abord la liste des stations trouvées avec leur période de
   données disponible — regarde si celles qui t'intéressent y sont.
4. Il génère ensuite un fichier JSON par station dans le dossier courant.
"""

import csv
import json
import re
from datetime import date
from collections import defaultdict

CHEMIN_FICHIER = r"C:\Users\PaulineJoly\agence du climat, le guichet des solutions\MOBILITE DECARBONEE - Documents\DONNEES\AUTRES_ANALYSES\CLIMAT\METEO_FRANCE\68\Q_68_previous-1950-2023_RR-T-Vent.csv"
SEPARATEUR = ";"
PERIODE_REF = (1976, 2005)
MIN_ANNEES = 50   # ignore les stations avec trop peu d'historique pour être utiles

# ----------------------------------------------------------------------
def vers_nombre(valeur):
    if valeur is None or str(valeur).strip() == "":
        return None
    try:
        return float(str(valeur).replace(",", "."))
    except ValueError:
        return None

def slugifier(nom):
    """'STRASBOURG-ENTZHEIM' -> 'strasbourg-entzheim' (utilisable comme nom de fichier/id)"""
    nom = nom.strip().lower()
    nom = re.sub(r"[^a-z0-9]+", "-", nom)
    return nom.strip("-")

def centile_95(valeurs):
    valeurs = sorted(v for v in valeurs if v is not None)
    if not valeurs:
        return None
    index = round(0.95 * (len(valeurs) - 1))
    return valeurs[index]

def moyenne(valeurs):
    valeurs = [v for v in valeurs if v is not None]
    return round(sum(valeurs) / len(valeurs), 2) if valeurs else None

def compter_jours_vague(jours_annee):
    jours_annee = sorted(jours_annee, key=lambda j: j["date"])
    total, run = 0, 0
    for j in jours_annee:
        if j["jour_chaud_ref"]:
            run += 1
        else:
            if run >= 3:
                total += run
            run = 0
    if run >= 3:
        total += run
    return total

# ----------------------------------------------------------------------
# 1) LECTURE + regroupement par station
# ----------------------------------------------------------------------
print(f"Lecture de {CHEMIN_FICHIER} ...")
with open(CHEMIN_FICHIER, encoding="utf-8", errors="replace") as f:
    lecteur = csv.DictReader(f, delimiter=SEPARATEUR)
    colonnes = lecteur.fieldnames
    print("Colonnes détectées :", colonnes)
    lignes_brutes = list(lecteur)

print(f"{len(lignes_brutes)} lignes lues.\n")

# Colonnes habituelles Météo-France : NUM_POSTE, NOM_USUEL identifient la station
jours_par_station = defaultdict(list)

for ligne in lignes_brutes:
    brut = ligne.get("AAAAMMJJ", "").strip()
    if len(brut) != 8:
        continue
    try:
        d = date(int(brut[0:4]), int(brut[4:6]), int(brut[6:8]))
    except ValueError:
        continue

    nom_station = (ligne.get("NOM_USUEL") or ligne.get("NUM_POSTE") or "STATION_INCONNUE").strip()

    tx = vers_nombre(ligne.get("TX"))
    tn = vers_nombre(ligne.get("TN"))
    tm = vers_nombre(ligne.get("TM"))
    if tm is None and tx is not None and tn is not None:
        tm = (tx + tn) / 2

    jours_par_station[nom_station].append({
        "date": d, "annee": d.year, "mois": d.month,
        "jour_annee": d.timetuple().tm_yday,
        "tx": tx, "tn": tn, "tm": tm,
    })

# ----------------------------------------------------------------------
# 2) APERÇU DES STATIONS TROUVÉES
# ----------------------------------------------------------------------
print("="*70)
print("STATIONS TROUVÉES DANS CE FICHIER")
print("="*70)
stations_retenues = []
for nom_station, jours in sorted(jours_par_station.items()):
    annees = sorted(set(j["annee"] for j in jours))
    nb_annees = len(annees)
    marque = "" if nb_annees >= MIN_ANNEES else "  (ignorée, historique trop court)"
    print(f"- {nom_station:35s} {annees[0]}-{annees[-1]}  ({nb_annees} années){marque}")
    if nb_annees >= MIN_ANNEES:
        stations_retenues.append(nom_station)

print(f"\n{len(stations_retenues)} station(s) seront traitées.\n")

# ----------------------------------------------------------------------
# 3) TRAITEMENT DE CHAQUE STATION RETENUE
# ----------------------------------------------------------------------
recap = []
for nom_station in stations_retenues:
    jours = jours_par_station[nom_station]

    # seuil vague de chaleur (centile 95 sur période de référence, par station)
    valeurs_par_jour_annee = defaultdict(list)
    for j in jours:
        if PERIODE_REF[0] <= j["annee"] <= PERIODE_REF[1]:
            valeurs_par_jour_annee[j["jour_annee"]].append(j["tm"])
    seuil_par_jour_annee = {ja: centile_95(v) for ja, v in valeurs_par_jour_annee.items()}
    for j in jours:
        seuil = seuil_par_jour_annee.get(j["jour_annee"])
        j["jour_chaud_ref"] = (seuil is not None and j["tm"] is not None and j["tm"] > seuil)

    jours_par_annee = defaultdict(list)
    for j in jours:
        jours_par_annee[j["annee"]].append(j)

    annees_stats = []
    for annee in sorted(jours_par_annee):
        jours_annee = jours_par_annee[annee]
        ete = [j for j in jours_annee if j["mois"] in (6, 7, 8)]
        annees_stats.append({
            "annee": annee,
            "tx_moy": moyenne([j["tx"] for j in jours_annee]),
            "tn_moy": moyenne([j["tn"] for j in jours_annee]),
            "ete_tm_moy": moyenne([j["tm"] for j in ete]),
            "jours_chauds_25": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 25),
            "jours_chauds_30": sum(1 for j in jours_annee if j["tx"] is not None and j["tx"] >= 30),
            "nuits_tropicales": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] >= 20),
            "jours_gel": sum(1 for j in jours_annee if j["tn"] is not None and j["tn"] < 0),
            "jours_vague_chaleur": compter_jours_vague(jours_annee),
        })

    slug = slugifier(nom_station)
    payload = {
        "station": nom_station.title(),
        "periode_reference": f"{PERIODE_REF[0]}-{PERIODE_REF[1]}",
        "source": "Météo-France",
        "projections": None,
        "annees": annees_stats,
    }
    nom_fichier = f"{slug}.json"
    with open(nom_fichier, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    recap.append((nom_station, slug, nom_fichier, len(annees_stats)))
    print(f"✅ {nom_fichier} généré ({len(annees_stats)} années) — id à utiliser dans communes.json : \"{slug}\"")

print("\n" + "="*70)
print("RÉCAPITULATIF — à reporter dans data/communes.json (liste 'stations')")
print("="*70)
for nom_station, slug, nom_fichier, n in recap:
    print(f'{{ "id": "{slug}", "nom": "{nom_station.title()}", "fichier": "data/{nom_fichier}" }}')

Lecture de C:\Users\PaulineJoly\agence du climat, le guichet des solutions\MOBILITE DECARBONEE - Documents\DONNEES\AUTRES_ANALYSES\CLIMAT\METEO_FRANCE\68\Q_68_previous-1950-2023_RR-T-Vent.csv ...
Colonnes détectées : ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'QRR', 'TN', 'QTN', 'HTN', 'QHTN', 'TX', 'QTX', 'HTX', 'QHTX', 'TM', 'QTM', 'TNTXM', 'QTNTXM', 'TAMPLI', 'QTAMPLI', 'TNSOL', 'QTNSOL', 'TN50', 'QTN50', 'DG', 'QDG', 'FFM', 'QFFM', 'FF2M', 'QFF2M', 'FXY', 'QFXY', 'DXY', 'QDXY', 'HXY', 'QHXY', 'FXI', 'QFXI', 'DXI', 'QDXI', 'HXI', 'QHXI', 'FXI2', 'QFXI2', 'DXI2', 'QDXI2', 'HXI2', 'QHXI2', 'FXI3S', 'QFXI3S', 'DXI3S', 'QDXI3S', 'HXI3S', 'QHXI3S', 'DRR', 'QDRR']
1049655 lignes lues.

STATIONS TROUVÉES DANS CE FICHIER
- ALGOLSHEIM SAD                      1986-1997  (12 années)  (ignorée, historique trop court)
- ALLAGOUTTES                         1997-2012  (16 années)  (ignorée, historique trop court)
- ALTENBERG                           1966-1985  (20 années)

✅ altkirch.json généré (52 années) — id à utiliser dans communes.json : "altkirch"
✅ bale-mulhouse.json généré (74 années) — id à utiliser dans communes.json : "bale-mulhouse"
✅ bits-les-thann.json généré (50 années) — id à utiliser dans communes.json : "bits-les-thann"
✅ colmar-inrae.json généré (52 années) — id à utiliser dans communes.json : "colmar-inrae"
✅ colmar-meyenheim.json généré (67 années) — id à utiliser dans communes.json : "colmar-meyenheim"
✅ geishouse-sapc.json généré (60 années) — id à utiliser dans communes.json : "geishouse-sapc"
✅ jebsheim.json généré (53 années) — id à utiliser dans communes.json : "jebsheim"
✅ kaysersberg.json généré (70 années) — id à utiliser dans communes.json : "kaysersberg"
✅ kiffis.json généré (74 années) — id à utiliser dans communes.json : "kiffis"
✅ kruth-mf.json généré (59 années) — id à utiliser dans communes.json : "kruth-mf"
✅ linthal-sapc.json généré (57 années) — id à utiliser dans communes.json : "linthal-sapc"
✅ mittlach-erbe.jso